In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import scipy.optimize as sco
import os
import warnings

warnings.filterwarnings('ignore')

def executar_markowitz_v6_atualizado(diretorio_data):
    print("=== EXECUTANDO MODELO DE MARKOWITZ V6 (PARAMETRIZAÇÃO ATUALIZADA) ===")
    
    # 1. Carregamento de Dados (Utilizando a Matriz de Prémio de Risco)
    # Aqui já temos Retornos - CDI e o filtro 2010-2025 aplicado
    caminho_er = os.path.join(diretorio_data, "matriz_premio_risco.csv")
    df_er = pd.read_csv(caminho_er, index_col='Data', parse_dates=True)
    
    # Separar Ativos do IBOV
    colunas_ativos = [col for col in df_er.columns if col != 'Premio_Mercado_IBOV']
    df_ativos = df_er[colunas_ativos]
    
    # Parâmetros Estatísticos Anualizados (252 dias)
    ret_esperados = df_ativos.mean() * 252
    cov_matrix = df_ativos.cov() * 252
    num_ativos = len(colunas_ativos)
    
    # 2. Funções de Otimização (Scipy SLSQP)
    def port_stats(weights):
        weights = np.array(weights)
        port_ret = np.sum(ret_esperados * weights)
        port_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
        return np.array([port_ret, port_vol, port_ret/port_vol])

    def min_sharpe(weights): return -port_stats(weights)[2]
    def min_vol(weights): return port_stats(weights)[1]

    # Restrições: Soma dos pesos = 1 | Limites: 0 a 1 (Long-only)
    cons = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bnds = tuple((0, 1) for x in range(num_ativos))
    
    print("2. Localizando Portfólios Ótimos...")
    # Máximo Sharpe
    opt_sharpe = sco.minimize(min_sharpe, num_ativos*[1./num_ativos], method='SLSQP', bounds=bnds, constraints=cons)
    # Mínima Variância
    opt_vol = sco.minimize(min_vol, num_ativos*[1./num_ativos], method='SLSQP', bounds=bnds, constraints=cons)
    
    # 3. Cálculo da Linha da Fronteira Eficiente (Delineamento Perfeito)
    print("3. Delineando a Linha da Fronteira Eficiente...")
    target_rets = np.linspace(ret_esperados.min(), ret_esperados.max(), 50)
    frontier_vols = []
    
    for tr in target_rets:
        cons_frontier = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1},
                         {'type': 'eq', 'fun': lambda x: port_stats(x)[0] - tr})
        res = sco.minimize(min_vol, num_ativos*[1./num_ativos], method='SLSQP', bounds=bnds, constraints=cons_frontier)
        frontier_vols.append(res['fun'])

    # 4. Simulação Monte Carlo (Para a nuvem de pontos decorativa)
    print("4. Gerando nuvem de carteiras aleatórias...")
    num_sim = 5000000
    sim_results = np.zeros((3, num_sim))
    for i in range(num_sim):
        w = np.random.random(num_ativos)
        w /= np.sum(w)
        stats = port_stats(w)
        sim_results[0,i], sim_results[1,i], sim_results[2,i] = stats[0], stats[1], stats[2]

    # 5. Visualização Interativa (Plotly - Estilo Script 6)
    fig = go.Figure()

    # Nuvem de Pontos
    fig.add_trace(go.Scatter(x=sim_results[1,:], y=sim_results[0,:], mode='markers',
                             marker=dict(color=sim_results[2,:], colorscale='Viridis', showscale=True, size=5),
                             name='Carteiras Aleatórias', opacity=0.3))

    # Linha da Fronteira
    fig.add_trace(go.Scatter(x=frontier_vols, y=target_rets, mode='lines', 
                             line=dict(color='black', width=3, dash='dash'), name='Fronteira Eficiente'))

    # Portfólios de Destaque
    res_s = port_stats(opt_sharpe.x)
    res_v = port_stats(opt_vol.x)

    fig.add_trace(go.Scatter(x=[res_s[1]], y=[res_s[0]], mode='markers', 
                             marker=dict(color='red', size=15, symbol='star'), name='Máximo Sharpe'))
    
    fig.add_trace(go.Scatter(x=[res_v[1]], y=[res_v[0]], mode='markers', 
                             marker=dict(color='green', size=15, symbol='star'), name='Mínima Variância'))

    fig.update_layout(title='Fronteira Eficiente Markowitz (136 Ativos - 2010 a 2025)',
                      xaxis_title='Volatilidade Anualizada', yaxis_title='Retorno em Excesso (vs CDI)',
                      template='plotly_white')
    
    #fig.show()

    # 6. Exibição da Composição (Igual ao Script 6)
    def exibir_composicao(weights, titulo):
        df_w = pd.DataFrame({'Ativo': colunas_ativos, 'Peso': weights})
        df_w = df_w[df_w['Peso'] > 0.01].sort_values(by='Peso', ascending=False)
        print(f"\n⭐ {titulo} ⭐")
        for index, row in df_w.iterrows():
            print(f"{row['Ativo']}: {row['Peso']:.2%}")

    exibir_composicao(opt_sharpe.x, "CARTEIRA MÁXIMO SHARPE")
    exibir_composicao(opt_vol.x, "CARTEIRA MÍNIMA VARIÂNCIA")

# Execução
pasta = r"C:\VSCodeWorkspace\TCC_Escrito\data"
executar_markowitz_v6_atualizado(pasta)

=== EXECUTANDO MODELO DE MARKOWITZ V6 (PARAMETRIZAÇÃO ATUALIZADA) ===
2. Localizando Portfólios Ótimos...
3. Delineando a Linha da Fronteira Eficiente...
4. Gerando nuvem de carteiras aleatórias...

⭐ CARTEIRA MÁXIMO SHARPE ⭐
CGAS5: 16.01%
WEGE3_: 11.91%
EQTL3: 11.81%
SAPR4: 11.67%
UNIP6: 11.63%
CLSC4: 7.72%
RADL3: 6.58%
FICT3: 6.54%
RCSL4: 3.19%
SLCE3: 3.04%
BEES3: 2.85%
KEPL3: 2.26%
PSSA3: 1.77%

⭐ CARTEIRA MÍNIMA VARIÂNCIA ⭐
AGRO3: 9.55%
COCE5: 9.22%
ISAE4: 8.17%
CGAS5: 7.67%
BEES3: 6.69%
EGIE3: 5.68%
CGRA4: 5.26%
VIVT3: 5.14%
CLSC4: 5.13%
KLBN4: 5.12%
ABEV3: 4.99%
WHRL4: 4.68%
ODPV3: 4.15%
PSSA3: 3.46%
SCAR3: 3.29%
TUPY3: 2.41%
SLCE3: 2.18%
RADL3: 1.82%
WEGE3_: 1.13%
